In [3]:
from optimizer_functions import *
import pandas as pd
from pyomo.opt import TerminationCondition, SolverStatus
from time import time
import matplotlib.pyplot as plt
import seaborn as sns
import pprint
import numpy as np
import math
import sys

# Parameters Calculation

In [4]:
# ── 1. load datasets ────────────────────────────────────────────────
# Load the trips data
df_raw = pd.read_csv('transit_departure_updated_energy.csv')
print(f"Trips before cleaning: {df_raw.shape[0]}")

df = process_transit_data(df_raw, tolerance=1, timestep_min=60)
print(f"Trips after cleaning: {df.shape[0]}")

#target_lines = ['1-1','1-3','1-4','1-9','1-11','1-13','1-16','1-18','1-19','1-22','1-25','1-28','1-29','1-32','1-33','1-36','1-39','1-51','1-52','1-53','1-54','1-55','1-56','1-57','1-58','1-59','1-61','1-64','1-65','1-70','1-72','1-74','1-76','1-79','1-80','1-81','1-82','1-84','1-85','1-86','1-88','1-90','1-91','1-92','1-93','1-94','1-96','1-107','1-133','1-136','1-185','1-400','1-401']  # Full set of lines
target_lines = ['1-1','1-4','1-11', '1-18', '1-25','1-54','1-60','1-70','1-72','1-80']  # Reduced set for quicker testing
#target_lines = target_lines[:10]
df = df[df["Line ID"].isin(target_lines)].copy()
#energy_price_data = pd.read_excel('tou_tariffs.xlsx', sheet_name=None)
#energy_price = energy_price_data['Sheet1']['Price'].to_list()
energy_price_data = pd.read_excel('tariffs_hydro.xlsx', sheet_name=None)
energy_price_L = energy_price_data['Sheet1']['L'].to_list()
energy_price_LG = energy_price_data['Sheet1']['LG'].to_list()
energy_price_L = resample_time_series(energy_price_L, original_delta=1, target_delta=1, method='average')
energy_price_LG = resample_time_series(energy_price_LG, original_delta=1, target_delta=1, method='average')
peak_price_L = 14.476
peak_price_LG = 15.963

# ── 2. save optimization dataset ─────────────────────────────────────
df.to_excel('optimization_dataset.xlsx', index=False)
print(f"Filtered trips: {df.shape[0]}")

Trips before cleaning: 3737
Trips after cleaning: 3629
Filtered trips: 661


In [5]:
# ── 3. extract data sets ───────────────────────────────────────────────
# Extract the necessary data from the DataFrame
start = df['Departure Step'].astype(int).to_list()
end   = df['Arrival Step'].astype(int).to_list()

# Define winter scenario factor (example: 30% more energy consumption)
winter_factor = 1.23  # Adjust this value based on your needs

# Apply winter factor to energy consumption
correction_factor = 1.02
gama_slow = df['Energy per timestep'].astype(float).to_list()
gama_slow = [energy * correction_factor for energy in gama_slow]
gama_fast = [energy * 0.95 for energy in gama_slow]
gama_slow_winter = [energy * winter_factor for energy in gama_slow]
gama_fast_winter = [energy * winter_factor * 0.95 for energy in gama_slow]

print(f"Average energy consumption (slow): {sum(gama_slow)/len(gama_slow):.2f} kWh")
print(f"Average energy consumption (fast): {sum(gama_fast)/len(gama_fast):.2f} kWh")
print(f"Average energy consumption (winter slow): {sum(gama_slow_winter)/len(gama_slow_winter):.2f} kWh")
print(f"Average energy consumption (winter fast): {sum(gama_fast_winter)/len(gama_fast_winter):.2f} kWh")

Average energy consumption (slow): 23.78 kWh
Average energy consumption (fast): 22.60 kWh
Average energy consumption (winter slow): 29.26 kWh
Average energy consumption (winter fast): 27.79 kWh


# Binary Search Parameters

In [ ]:
# ── 4. energy consumption per route (boxplot) ───────────────────────────────
df_plot = df[["Line ID", "Avg Energy (kWh/km)"]].copy()
df_plot["slow_mild"] = df_plot["Avg Energy (kWh/km)"] * correction_factor
df_plot["fast_mild"] = df_plot["Avg Energy (kWh/km)"] * 0.95 * correction_factor
df_plot["slow_winter"] = df_plot["Avg Energy (kWh/km)"] * winter_factor * correction_factor
df_plot["fast_winter"] = df_plot["Avg Energy (kWh/km)"] * winter_factor * 0.95 * correction_factor

long_df = df_plot.melt(
    id_vars=["Line ID"],
    value_vars=["slow_mild", "fast_mild", "slow_winter", "fast_winter"],
    var_name="Scenario",
    value_name="Energy_kWh_km"
 )

scenario_labels = {
    "slow_mild": "Slow (mild)",
    "fast_mild": "Fast (mild)",
    "slow_winter": "Slow (winter)",
    "fast_winter": "Fast (winter)"
}
long_df["Scenario"] = long_df["Scenario"].map(scenario_labels)

# Scenario averages (kWh/km)
avg_series = long_df.groupby("Scenario")["Energy_kWh_km"].mean().round(3)
print("Averages (kWh/km)")
for scenario, value in avg_series.items():
    print(f"{scenario}: {value}")

route_order = sorted(long_df["Line ID"].unique())
plt.figure(figsize=(14, 6), dpi=300)
ax = sns.boxplot(
    data=long_df,
    x="Line ID",
    y="Energy_kWh_km",
    hue="Scenario",
    order=route_order,
    showfliers=False
 )

# Add dashed separators between routes
for idx in range(len(route_order) - 1):
    ax.axvline(idx + 0.5, color="gray", linestyle="--", linewidth=0.6, alpha=0.6)

# Horizontal dashed lines for y-axis readability
ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5)

plt.xticks(rotation=60, ha="right")
plt.xlabel("Route (Line ID)")
plt.ylabel("Energy (kWh/km)")
#plt.title("Energy consumption per route by strategy and season")
plt.legend(title="Scenario", loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. binary search for minimum fleet size and chargers ───────────────────────────────
def binary_search(bus_counts, battery_sizes, charger_powers):
    """
    Modified binary search to find minimum feasible fleet size and charging infrastructure
    """
    def check_feasibility(n_buses, n_chargers, find_optimal=False):
        """Test if solution is feasible with given fleet size"""
        try:
            start_time = time()
            test_C_bat, test_alpha = build_fleet_and_chargers(
                [n_buses], battery_sizes,
                [n_chargers], charger_powers
            )
            
            test_model = optimization(start, end, test_alpha, gama, test_C_bat, energy_price,
                                E_0=0.2, E_min=0.2, E_max=1.00, E_end=0.2, 
                                delta_t=1, variable_power=True)
            
            solver = SolverFactory('gurobi')
            if find_optimal:
                solver.options['TimeLimit'] = 3600  # 1 hour for final optimization
                solver.options['MIPGap'] = 0.05     # 5% gap for final optimization
            else:
                solver.options['TimeLimit'] = 600   # 10 minutes for feasibility check
                solver.options['MIPGap'] = 0.9
                solver.options['SolutionLimit'] = 1
            
            results = solver.solve(test_model, tee=False)
            solve_time = time() - start_time

            # ────────────────────────────────────────────────────────────────
            # Expanded feasibility condition
            # ────────────────────────────────────────────────────────────────
            status = results.solver.status
            term   = results.solver.termination_condition

            if (
                term == TerminationCondition.optimal
                or term == TerminationCondition.feasible
                or (status == SolverStatus.aborted and hasattr(test_model, "solutions"))
                or (status == SolverStatus.ok and hasattr(test_model, "solutions"))
            ):
                print(f"Solution time: {solve_time:.2f} seconds (feasible)")
                return True, test_model if find_optimal else None

            print(f"Solution time: {solve_time:.2f} seconds (infeasible)")
            return False, None
        except Exception as e:
            print(f"Error occurred during optimization: {e}")
            return False, None
    
    total_start_time = time()

    # Phase 1: Find minimum number of buses
    print("Phase 1: Finding minimum number of buses...\n\n")
    
    # Try initial bus count
    initial_buses = bus_counts[0]
    is_feasible_sol, model = check_feasibility(initial_buses, initial_buses)
    
    if is_feasible_sol:
        right_bus = initial_buses
        left_bus = initial_buses // 2
    else:
        # Try increasing the buses by 25% until feasible
        current_buses = initial_buses * 2
        while True:
            print(f"Initial {initial_buses} buses infeasible, trying {current_buses} buses...")
            is_feasible_sol, model = check_feasibility(current_buses, current_buses)
            if is_feasible_sol:
                print(f"Feasible solution found with {current_buses} buses")
                break
            # If not feasible, increment by 25%
            next_buses = int(current_buses * 1.25)
            # Prevent infinite loop if increment does not increase bus count
            if next_buses == current_buses:
                print("Problem infeasible even after repeated increments")
                return None, None, None
            current_buses = next_buses

        right_bus = current_buses
        left_bus = initial_buses

    min_buses = right_bus
    feasible_model = model
    
    # Binary search for minimum buses
    while right_bus - left_bus > 1:
        mid_bus = (left_bus + right_bus) // 2
        print(f"\nTesting with {mid_bus} buses...")
        
        is_feasible_sol, model = check_feasibility(mid_bus, mid_bus)
        if is_feasible_sol:
            min_buses = mid_bus
            right_bus = mid_bus
            feasible_model = model
            print(f"Feasible solution found with {mid_bus} buses")
        else:
            left_bus = mid_bus
            print(f"Infeasible solution with {mid_bus} buses")
    
    # Phase 2: Find minimum number of chargers
    print(f"\n\nPhase 2: Finding minimum chargers for {min_buses} buses...")
    
    current_chargers = min_buses
    min_chargers = current_chargers
    step_size = current_chargers // 2
    
    while step_size > 0:
        test_chargers = current_chargers - step_size
        print(f"\nTesting with {min_buses} buses and {test_chargers} chargers...")
        
        is_feasible_sol, model = check_feasibility(min_buses, test_chargers)
        if is_feasible_sol:
            min_chargers = test_chargers
            current_chargers = test_chargers
            feasible_model = model
            print(f"Feasible solution found with {test_chargers} chargers")
        else:
            print(f"Infeasible solution with {test_chargers} chargers")
        
        step_size = step_size // 2
    
    print("\nFinal step: Solving for optimality with minimum fleet size...")
    _, optimal_model = check_feasibility(min_buses, min_chargers, find_optimal=True)
    
    total_time = time() - total_start_time
    print(f"\nTotal binary search time: {total_time:.2f} seconds")
    return min_buses, min_chargers, optimal_model


In [ ]:
# ── 5b. scenario-aware runner (keeps binary_search logic) ─────────────────────
def run_binary_search(initial_buses, strategy, season):
    """
    Run binary_search with scenario inputs without changing its internal logic.
    """
    strategy = strategy.lower().strip()
    season = season.lower().strip()
    if strategy not in ("slow", "fast"):
        raise ValueError("strategy must be 'slow' or 'fast'")
    if season not in ("mild", "winter"):
        raise ValueError("season must be 'mild' or 'winter'")
    
    # Set scenario-specific parameters
    if strategy == "slow":
        battery_sizes = [300]
        charger_powers = [100]
        if season == "mild":
            scenario_energy_price = energy_price_L
            scenario_gama = gama_slow
        else:
            scenario_energy_price = energy_price_LG
            scenario_gama = gama_slow_winter
    else:
        battery_sizes = [180]
        charger_powers = [400]
        if season == "mild":
            scenario_energy_price = energy_price_LG
            scenario_gama = gama_fast
        else:
            scenario_energy_price = energy_price_LG
            scenario_gama = gama_fast_winter
    
    # Make expected globals available to binary_search without changing its code
    global energy_price, gama
    energy_price = scenario_energy_price
    gama = scenario_gama
    bus_counts = [initial_buses]
    return binary_search(bus_counts, battery_sizes, charger_powers)

In [ ]:
# ── 5c. direct optimal value routine ─────────────────────────────────────────
def solve_with_fixed_fleet(n_buses, n_chargers, strategy, season, time_limit=3600, mip_gap=0.05, tee=False):
    """
    Solve for optimal objective with fixed buses/chargers and scenario inputs.
    Saves the output file with the scenario-specific filename.
    """
    strategy = strategy.lower().strip()
    season = season.lower().strip()
    if strategy not in ("slow", "fast"):
        raise ValueError("strategy must be 'slow' or 'fast'")
    if season not in ("mild", "winter"):
        raise ValueError("season must be 'mild' or 'winter'")
    
    if strategy == "slow":
        battery_sizes = [350]
        charger_powers = [100]
        vehicle_unit_cost = 1_200_000
        charger_unit_cost = 180_000
        implementation_unit_cost = 70_000
        if season == "mild":
            scenario_energy_price = energy_price_L
            scenario_gama = gama_slow
            peak_price = peak_price_L
            filename = "optimization_slowcharging_summary.xlsx"
        else:
            scenario_energy_price = energy_price_LG
            scenario_gama = gama_slow_winter
            peak_price = peak_price_LG
            filename = "optimization_slowcharging_winter_summary.xlsx"
    else:
        battery_sizes = [180]
        charger_powers = [400]
        vehicle_unit_cost = 850_000
        charger_unit_cost = 300_000
        implementation_unit_cost = 90_000
        if season == "mild":
            scenario_energy_price = energy_price_LG
            scenario_gama = gama_fast
            peak_price = peak_price_LG
            filename = "optimization_fastcharging_summary.xlsx"
        else:
            scenario_energy_price = energy_price_LG
            scenario_gama = gama_fast_winter
            peak_price = peak_price_LG
            filename = "optimization_fastcharging_winter_summary.xlsx"
    
    test_C_bat, test_alpha = build_fleet_and_chargers(
        [n_buses], battery_sizes,
        [n_chargers], charger_powers
    )
    model = optimization(
        start, end, test_alpha, scenario_gama, test_C_bat, scenario_energy_price,
        E_0=0.2, E_min=0.2, E_max=1.00, E_end=0.2,
        delta_t=1, variable_power=True
    )
    solver = SolverFactory('gurobi')
    solver.options['TimeLimit'] = time_limit
    solver.options['MIPGap'] = mip_gap
    results = solver.solve(model, tee=tee)
    status = results.solver.status
    term = results.solver.termination_condition
    if term in (TerminationCondition.optimal, TerminationCondition.feasible) or status == SolverStatus.ok:
        export_all_to_excel(
            model,
            vehicle_unit_cost=vehicle_unit_cost,
            charger_unit_cost=charger_unit_cost,
            implementation_unit_cost=implementation_unit_cost,
            peak_power_price_per_kw_day=peak_price,
            years_for_reference=10,
            filename=filename
        )
        return model, pyo.value(model.obj), filename
    return None, None, filename

# Run Model

In [ ]:
solve_with_fixed_fleet(n_buses=104, n_chargers=58, strategy='fast', season='winter', time_limit=800, mip_gap=0.05)

In [ ]:
# ─────────────── CAPEX + OPEX (from files) ───────────────
files = {
    "fast_summer": "optimization_fastcharging_summary.xlsx",
    "fast_winter": "optimization_fastcharging_winter_summary.xlsx",
    "slow_summer": "optimization_slowcharging_summary.xlsx",
    "slow_winter": "optimization_slowcharging_winter_summary.xlsx",
}
labels = {
    "slow_summer": "Slow charging (mild)",
    "slow_winter": "Slow charging (winter)",
    "fast_summer": "Fast charging (mild)",
    "fast_winter": "Fast charging (winter)",
}
scenario_order = [labels[key] for key in files.keys()]

capex_rows = []
for key, path in files.items():
    capex_df = pd.read_excel(path, sheet_name="Costs_CAPEX")
    capex_df = capex_df[capex_df["Category"].isin(["Fleet CAPEX", "Charger CAPEX", "Implementation"])].copy()
    capex_df["MonthlyValue"] = capex_df["Total [€]"]
    capex_sum = capex_df.groupby("Category", as_index=False)["MonthlyValue"].sum()
    capex_sum["Scenario"] = labels[key]
    capex_rows.append(capex_sum)
capex_plot = pd.concat(capex_rows, ignore_index=True)

opex_rows = []

def _get_daily_cost_any(df, patterns, default=0.0):
    mask = df["Category"].str.contains("|".join(patterns), case=False, na=False)
    if not mask.any():
        return float(default)
    value = df.loc[mask, "Value [€]"].iloc[0]
    return float(value)

for key, path in files.items():
    opex_df = pd.read_excel(path, sheet_name="Costs_OPEX_Daily")
    
    # Clean column names
    opex_df.columns = opex_df.columns.str.strip()
    opex_df["Category"] = opex_df["Category"].str.strip()
    
    # Extract daily values
    energy_daily = _get_daily_cost_any(opex_df, ["Energy"], default=0.0)
    power_daily = _get_daily_cost_any(opex_df, ["Power", "Peak"], default=0.0)
    total_daily = _get_daily_cost_any(opex_df, ["OPEX Total", "Total OPEX", "OPEX"], default=0.0)
    if power_daily == 0.0 and total_daily > 0.0:
        power_daily = max(total_daily - energy_daily, 0.0)
    if energy_daily == 0.0 and total_daily > 0.0 and power_daily > 0.0:
        energy_daily = max(total_daily - power_daily, 0.0)
    
    # Convert to monthly
    energy_monthly = energy_daily * 30
    power_monthly = power_daily * 30
    
    # Correct total
    opex_monthly = energy_monthly + power_monthly
    
    opex_rows.extend([
        {"Scenario": labels[key], "Metric": "Contracted power", "Value": power_monthly},
        {"Scenario": labels[key], "Metric": "Energy", "Value": energy_monthly},
        {"Scenario": labels[key], "Metric": "OPEX", "Value": opex_monthly},
    ])

opex_plot = pd.DataFrame(opex_rows)


import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

# ─────────────── CAPEX PLOT (categories on x) ───────────────
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=capex_plot,
    x="Category",
    y="MonthlyValue",
    hue="Scenario",
    hue_order=scenario_order,
    ax=ax,
    edgecolor="black"
 )

# ─────────────── Y-AXIS IN MILLIONS ───────────────
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"{x/1e6:.0f} M")
 )

ax.set_ylabel("Total Cost (CAD$)")
ax.set_xlabel("")
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.legend(title="")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


# ─────────────── OPEX PLOT (categories on x) ───────────────
metric_order = ["Contracted power", "Energy", "OPEX"]
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=opex_plot,
    x="Metric",
    y="Value",
    hue="Scenario",
    order=metric_order,
    hue_order=scenario_order,
    ax=ax,
    edgecolor="black"
 )
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"{x/1e3:.0f} K")
 )
ax.set_ylabel("Monthly Cost (CAD$)")
ax.set_xlabel("")
#ax.set_title("OPEX (monthly)")
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.legend(title="")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Temperature visualization style
# ─────────────── DATA ───────────────
data = {
    "Month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "High": [-5, -3, 3, 12, 19, 24, 26, 25, 20, 13, 6, -1],
    "High_err": [4, 4, 4, 3, 3, 2, 2, 2, 2, 3, 3, 4],
    "Avg": [-9, -7, -1, 7, 14, 19, 21, 21, 16, 9, 3, -5],
    "Avg_err": [4, 4, 3, 3, 2, 2, 2, 2, 2, 3, 3, 4],
    "Low": [-13, -11, -5, 3, 10, 15, 17, 16, 12, 6, -1, -8],
    "Low_err": [4, 4, 4, 3, 2, 2, 2, 2, 2, 3, 3, 4]
}
df = pd.DataFrame(data)
x = np.arange(len(df["Month"]))

# ─────────────── STYLE ───────────────
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "axes.facecolor": "white"
})
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(12, 6))

# ─────────────── TEMPERATURE CURVES ───────────────
ax.plot(x, df["High"], color="#E15759", linewidth=2.2, label="High (°C)")
ax.fill_between(x, df["High"] - df["High_err"], df["High"] + df["High_err"], color="#E15759", alpha=0.15)

ax.plot(x, df["Avg"], color="#1D86D6", linewidth=2.2, label="Average (°C)")
ax.fill_between(x, df["Avg"] - df["Avg_err"], df["Avg"] + df["Avg_err"], color="#1D86D6", alpha=0.15)

ax.plot(x, df["Low"], color="#4C72B0", linewidth=2.2, label="Low (°C)")
ax.fill_between(x, df["Low"] - df["Low_err"], df["Low"] + df["Low_err"], color="#4C72B0", alpha=0.15)

# ─────────────── SEASONAL SHADING ───────────────
# Cold: Dec–Mar
ax.axvspan(-0.5, 2.5, color="#C6DBEF", alpha=0.3)
ax.axvspan(10.5, 11.5, color="#C6DBEF", alpha=0.3)
# Mild: Apr–Nov
ax.axvspan(2.5, 10.5, color="#FEE0D2", alpha=0.3)

# ─────────────── SEASON LABELS ───────────────
ax.text(1, 38, "cold", ha="center", fontsize=12, fontweight="bold", color="#08306B")
ax.text(6.5, 38, "mild", ha="center", fontsize=12, fontweight="bold", color="#67000D")
ax.text(11, 38, "cold", ha="center", fontsize=12, fontweight="bold", color="#08306B")

# ─────────────── HIGHLIGHT KEY MONTHS ───────────────
for i, month in enumerate(df["Month"]):
    if month in ["Jan", "Mar", "May", "Jul", "Sep", "Dec"]:
        ax.scatter(x[i], df["High"][i], color="#E15759", s=20)
        ax.text(x[i], df["High"][i] + 2, f"{df['High'][i]}°C", ha="center", fontsize=10, weight="bold", color="#E15759")
        ax.scatter(x[i], df["Low"][i], color="#4C72B0", s=20)
        ax.text(x[i], df["Low"][i] - 3, f"{df['Low'][i]}°C", ha="center", fontsize=10, weight="bold", color="#4C72B0")

# ─────────────── AXES FORMATTING ───────────────
ax.set_xticks(x)
ax.set_xticklabels(df["Month"])
ax.set_xlabel("Month")
ax.set_ylabel("Temperature (°C)")
ax.set_ylim(-25, 45)
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.legend(frameon=True, loc="upper left")

plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────── STYLE ───────────────
plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "axes.facecolor": "white",
    "axes.edgecolor": "black"
})
sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.linewidth": 1.2})

# ─────────────── INPUT DATA (Million €) ───────────────
data = {
    "Strategy": ["Fast Charging", "Slow Charging"],
    "CAPEX (€)": [111.02e6, 126.10e6],
    "Power (10 yr) (€)": [34.60e6, 8.70e6],      # demand component (approx. from OPEX structure)
    "Energy (10 yr) (€)": [4.15e6, 3.90e6]       # energy component
}

plot_df = pd.DataFrame(data)
plot_df["TCO (€)"] = plot_df[["CAPEX (€)", "Power (10 yr) (€)", "Energy (10 yr) (€)"]].sum(axis=1)

# ─────────────── COMPONENTS FOR STACKED BARS ───────────────
components = [
    "CAPEX (€)",
    "Power (10 yr) (€)",
    "Energy (10 yr) (€)"
]

colors = [
    "#4C72B0",   # CAPEX
    "#E15759",   # Power (dominant OPEX driver)
    "#F28E2B"    # Energy
]

# ─────────────── PLOT ───────────────
fig, ax = plt.subplots(figsize=(14, 10))
bottom = np.zeros(len(plot_df))

for comp, color in zip(components, colors):
    ax.bar(
        plot_df["Strategy"],
        plot_df[comp],
        bottom=bottom,
        label=comp.replace(" (€)", ""),
        color=color,
        edgecolor="black",
        linewidth=1.1
    )
    bottom += plot_df[comp].values

# ─────────────── CAPEX/OPEX COMPUTATION ───────────────
plot_df["Total CAPEX"] = plot_df["CAPEX (€)"]
plot_df["Total OPEX"] = plot_df[["Power (10 yr) (€)", "Energy (10 yr) (€)"]].sum(axis=1)

plot_df["CAPEX %"] = (plot_df["Total CAPEX"] / plot_df["TCO (€)"] * 100).round(1)
plot_df["OPEX %"] = (plot_df["Total OPEX"] / plot_df["TCO (€)"] * 100).round(1)

# ─────────────── PERCENTAGE BRACKETS ───────────────
for i in range(len(plot_df)):
    capex_val = plot_df.loc[i, "Total CAPEX"]
    opex_val = plot_df.loc[i, "Total OPEX"]
    total = plot_df.loc[i, "TCO (€)"]

    # CAPEX bracket
    ax.annotate(
        "",
        xy=(i, capex_val),
        xytext=(i, 0),
        arrowprops=dict(arrowstyle="|-|", lw=1.4, color="black")
    )
    ax.text(
        i - 0.1,
        capex_val / 2,
        f"CAPEX {plot_df.loc[i, 'CAPEX %']}%",
        ha="right",
        va="center",
        fontsize=12,
        fontweight="bold"
    )

    # OPEX bracket
    ax.annotate(
        "",
        xy=(i, total),
        xytext=(i, capex_val),
        arrowprops=dict(arrowstyle="|-|", lw=1.4, color="black")
    )
    ax.text(
        i - 0.1,
        capex_val + opex_val / 2,
        f"OPEX {plot_df.loc[i, 'OPEX %']}%",
        ha="right",
        va="center",
        fontsize=12,
        fontweight="bold"
    )

# ─────────────── TCO LABELS ───────────────
for i, total in enumerate(plot_df["TCO (€)"]):
    ax.text(
        i,
        total * 1.03,
        f"TCO: {total/1e6:.1f} M€",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", boxstyle="round,pad=0.3")
    )

# ─────────────── STYLE & AXES ───────────────
ax.set_ylabel("Cost (€)", labelpad=12)
ax.set_xlabel("")
ax.set_ylim(0, plot_df["TCO (€)"].max() * 1.20)
ax.grid(axis="y", linestyle="--", alpha=0.85, linewidth=1.2)

ax.legend(frameon=True, loc="upper right", bbox_to_anchor=(0.98, 0.98), ncol=1)

plt.subplots_adjust(left=0.08, right=0.95, top=0.97, bottom=0.08)
plt.show()



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================================
# USER INPUT DATA
# (values in €)
# ==========================================================

years = 10

data = {
    "Strategy": ["Fast Charging", "Slow Charging"],
    
    # ---------- CAPEX (fill or adjust if needed) ----------
    # Total CAPEX winter sizing:
    # Fast = 111.02 M€
    # Slow = 126.10 M€
    #
    # If you do not know the split yet, you can keep the proportions
    # or replace later.
    
    "Fleet (€)": [
        95.0e6,   # <-- adjust if needed
        108.0e6   # <-- adjust if needed
    ],
    
    "Charger (€)": [
        10.5e6,   # <-- adjust if needed
        12.5e6    # <-- adjust if needed
    ],
    
    "Implementation (€)": [
        5.52e6,   # <-- adjust if needed
        5.60e6    # <-- adjust if needed
    ],
    
    # ---------- OPEX (10-year totals) ----------
    # From previous calculations (8 mild + 4 winter)
    
    "Power (10 yr) (€)": [
        34.60e6,   # Fast
        8.70e6     # Slow
    ],
    
    "Energy (10 yr) (€)": [
        4.15e6,    # Fast
        3.90e6     # Slow
    ]
}

plot_df = pd.DataFrame(data)

# ==========================================================
# STYLE
# ==========================================================
plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "axes.facecolor": "white",
    "axes.edgecolor": "black"
})
sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.linewidth": 1.2})

# ==========================================================
# COMPUTATIONS
# ==========================================================
plot_df["Total CAPEX"] = plot_df[
    ["Fleet (€)", "Charger (€)", "Implementation (€)"]
].sum(axis=1)

plot_df["Total OPEX"] = plot_df[
    ["Power (10 yr) (€)", "Energy (10 yr) (€)"]
].sum(axis=1)

plot_df["TCO (€)"] = plot_df["Total CAPEX"] + plot_df["Total OPEX"]

plot_df["CAPEX %"] = (plot_df["Total CAPEX"] / plot_df["TCO (€)"] * 100).round(1)
plot_df["OPEX %"] = (plot_df["Total OPEX"] / plot_df["TCO (€)"] * 100).round(1)

# ==========================================================
# PLOT
# ==========================================================
components = [
    "Fleet (€)",
    "Charger (€)",
    "Implementation (€)",
    "Power (10 yr) (€)",
    "Energy (10 yr) (€)"
]

colors = [
    "#4C72B0",
    "#1D86D6",
    "#76B7B2",
    "#E15759",
    "#F28E2B"
]

fig, ax = plt.subplots(figsize=(14, 10))
bottom = np.zeros(len(plot_df))

for comp, color in zip(components, colors):
    ax.bar(
        plot_df["Strategy"],
        plot_df[comp],
        bottom=bottom,
        label=comp.replace(" (€)", ""),
        color=color,
        edgecolor="black",
        linewidth=1.1
    )
    bottom += plot_df[comp].values

# ---------- CAPEX / OPEX brackets ----------
for i in range(len(plot_df)):
    capex_val = plot_df.loc[i, "Total CAPEX"]
    opex_val = plot_df.loc[i, "Total OPEX"]
    total = plot_df.loc[i, "TCO (€)"]

    ax.annotate("", xy=(i, capex_val), xytext=(i, 0),
                arrowprops=dict(arrowstyle="|-|", lw=1.4, color="black"))
    ax.text(i - 0.1, capex_val/2,
            f"CAPEX {plot_df.loc[i,'CAPEX %']}%",
            ha="right", va="center", fontweight="bold")

    ax.annotate("", xy=(i, total), xytext=(i, capex_val),
                arrowprops=dict(arrowstyle="|-|", lw=1.4, color="black"))
    ax.text(i - 0.1, capex_val + opex_val/2,
            f"OPEX {plot_df.loc[i,'OPEX %']}%",
            ha="right", va="center", fontweight="bold")

# ---------- TCO labels ----------
for i, total in enumerate(plot_df["TCO (€)"]):
    ax.text(
        i,
        total * 1.03,
        f"TCO: {total/1e6:.1f} M€",
        ha="center",
        va="bottom",
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", boxstyle="round,pad=0.3")
    )

# ---------- Axis ----------
ax.set_ylabel("Cost (€)")
ax.set_ylim(0, plot_df["TCO (€)"].max() * 1.20)
ax.grid(axis="y", linestyle="--", alpha=0.85)
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────── EXTRA PLOTS: COST WATERFALL + BUS GANTT ───────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

files = {
    "fast_summer": "optimization_fastcharging_summary.xlsx",
    "fast_winter": "optimization_fastcharging_winter_summary.xlsx",
    "slow_summer": "optimization_slowcharging_summary.xlsx",
    "slow_winter": "optimization_slowcharging_winter_summary.xlsx",
}
labels = {
    "slow_summer": "Slow charging (mild)",
    "slow_winter": "Slow charging (winter)",
    "fast_summer": "Fast charging (mild)",
    "fast_winter": "Fast charging (winter)",
}

def _get_daily_cost_any(df, patterns, default=0.0):
    mask = df["Category"].str.contains("|".join(patterns), case=False, na=False)
    if not mask.any():
        return float(default)
    value = df.loc[mask, "Value [€]"].iloc[0]
    return float(value)

def _build_cost_components(path, years=10):
    capex_df = pd.read_excel(path, sheet_name="Costs_CAPEX")
    capex_df["Category"] = capex_df["Category"].str.strip()
    capex_df = capex_df[capex_df["Category"].isin(["Fleet CAPEX", "Charger CAPEX", "Implementation"])].copy()
    capex_sum = capex_df.groupby("Category", as_index=False)["Total [€]"].sum()
    capex_map = dict(zip(capex_sum["Category"], capex_sum["Total [€]"]))

    opex_df = pd.read_excel(path, sheet_name="Costs_OPEX_Daily")
    opex_df.columns = opex_df.columns.str.strip()
    opex_df["Category"] = opex_df["Category"].str.strip()
    energy_daily = _get_daily_cost_any(opex_df, ["Energy"], default=0.0)
    power_daily = _get_daily_cost_any(opex_df, ["Power", "Peak"], default=0.0)
    total_daily = _get_daily_cost_any(opex_df, ["OPEX Total", "Total OPEX", "OPEX"], default=0.0)
    if power_daily == 0.0 and total_daily > 0.0:
        power_daily = max(total_daily - energy_daily, 0.0)
    if energy_daily == 0.0 and total_daily > 0.0 and power_daily > 0.0:
        energy_daily = max(total_daily - power_daily, 0.0)
    energy_total = energy_daily * 365 * years
    power_total = power_daily * 365 * years

    return {
        "Fleet CAPEX": capex_map.get("Fleet CAPEX", 0.0),
        "Charger CAPEX": capex_map.get("Charger CAPEX", 0.0),
        "Implementation": capex_map.get("Implementation", 0.0),
        f"Energy ({years} yr)": energy_total,
        f"Contracted power ({years} yr)": power_total,
    }

# ─────────────── COST WATERFALL (by scenario) ───────────────
years = 10
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=300)
axes = axes.flatten()
scenario_keys = list(files.keys())

for ax, scen in zip(axes, scenario_keys):
    comp = _build_cost_components(files[scen], years=years)
    labels_order = list(comp.keys())
    values = [comp[k] for k in labels_order]
    cum = np.cumsum([0] + values[:-1])
    colors = ["#4C72B0", "#1D86D6", "#76B7B2", "#F28E2B", "#E15759"]

    ax.bar(labels_order, values, bottom=cum, color=colors[: len(values)], edgecolor="black")
    ax.set_title(labels[scen])
    ax.set_xticklabels(labels_order, rotation=45, ha="right")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x/1e6:.1f} M"))
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    ax.set_ylabel("Total Cost (CAD$)")

plt.tight_layout()
plt.show()


# Sensitivity Analysis

In [7]:
import pyomo.environ as pyo
import pandas as pd

def extract_kpis(model, peak_price_per_kw_day):
    dt = pyo.value(model.delta_t)
    
    # Energy and Power
    total_energy_kwh = sum(pyo.value(model.w_buy[t]) for t in model.T)
    peak_power_kw = max(pyo.value(model.w_buy[t]) / dt for t in model.T)
    
    # Financials
    energy_cost = sum(pyo.value(model.P[t]) * pyo.value(model.w_buy[t]) for t in model.T)
    power_cost = peak_power_kw * peak_price_per_kw_day
    daily_opex = energy_cost + (power_cost / 30) # Distributing monthly power charge to a daily view
    
    # Efficiency & Utilization
    total_trips = len(model.I)
    avg_kwh_per_trip = total_energy_kwh / total_trips if total_trips > 0 else 0
    
    charger_slots = len(model.N) * len(model.T)
    charger_usage = sum(pyo.value(model.x[k,n,t]) for k in model.K for n in model.N for t in model.T)
    
    return {
        "total_energy_kWh": round(total_energy_kwh, 2),
        "peak_power_kW": round(peak_power_kw, 2),
        "daily_opex": round(daily_opex, 2),
        "avg_kWh_per_trip": round(avg_kwh_per_trip, 3),
        "charger_utilization": round(charger_usage / charger_slots, 3),
        "min_soc_reached": round(min(pyo.value(model.e[k,t]/model.C_bat[k]) for k in model.K for t in model.T), 3)
    }

In [8]:
import pyomo.environ as pyo
import pandas as pd
from pyomo.opt import SolverFactory, TerminationCondition

def run_comprehensive_sensitivity(
    base_params,          
    variations,           # e.g., [0.8, 0.9, 1.1, 1.2] (exclude 1.0)
    weight_penalty=0.15,  
    peak_price=15.963,    
    time_limit=600
):
    results_list = []
    
    # --- 1. RUN BASELINE ONCE (Variation = 1.0) ---
    print("--- Running BASELINE (100%) ---")
    model_base = optimization(
        base_params['start'], base_params['end'], 
        base_params['alpha'], base_params['gamma'], base_params['C_bat'], 
        base_params['P'], delta_t=base_params.get('delta_t', 1.0),
        variable_power=True
    )
    
    solver = SolverFactory("gurobi")
    solver.options["TimeLimit"] = time_limit
    res_base = solver.solve(model_base)
    
    if res_base.solver.termination_condition in (TerminationCondition.optimal, TerminationCondition.feasible):
        base_kpis = extract_kpis(model_base, peak_price)
        base_status = "Success"
    else:
        base_kpis = {k: None for k in ["total_energy_kWh", "peak_power_kW", "daily_opex", 
                                      "avg_kWh_per_trip", "charger_utilization", "min_soc_reached"]}
        base_status = "Failed"

    results_list.append({
        "Parameter_Tested": "Baseline",
        "Variation": 1.0,
        "Weight_Impact": 1.0,
        "Status": base_status,
        **base_kpis
    })

    # --- 2. RUN SENSITIVITY LOOPS (Exclude 1.0) ---
    test_scenarios = {
        'Battery Capacity': 'C_bat',
        'Charger Power': 'alpha',
        'Consumption Rate': 'gamma'
    }

    # Clean variations list to avoid re-running 1.0
    active_variations = [v for v in variations if v != 1.0]

    for label, param_key in test_scenarios.items():
        for v in active_variations:
            # Fresh copies
            current_alpha = list(base_params['alpha'])
            current_gamma = list(base_params['gamma'])
            current_C_bat = list(base_params['C_bat'])
            
            b_factor, p_factor, c_factor = 1.0, 1.0, 1.0
            
            if param_key == 'C_bat':
                b_factor = v
                c_factor = 1.0 + (v - 1.0) * weight_penalty
            elif param_key == 'alpha':
                p_factor = v
            elif param_key == 'gamma':
                c_factor = v

            current_C_bat = [val * b_factor for val in current_C_bat]
            current_alpha = [val * p_factor for val in current_alpha]
            current_gamma = [val * c_factor for val in current_gamma]

            print(f"--- Running: {label} at {v*100:.0f}% ---")

            model = optimization(
                base_params['start'], base_params['end'], 
                current_alpha, current_gamma, current_C_bat, base_params['P'],
                delta_t=base_params.get('delta_t', 1.0),
                variable_power=True
            )

            results = solver.solve(model)

            if results.solver.termination_condition in (TerminationCondition.optimal, TerminationCondition.feasible):
                kpis = extract_kpis(model, peak_price)
                status = "Success"
            else:
                kpis = {k: None for k in ["total_energy_kWh", "peak_power_kW", "daily_opex", 
                                          "avg_kWh_per_trip", "charger_utilization", "min_soc_reached"]}
                status = "Infeasible/Failed"

            results_list.append({
                "Parameter_Tested": label,
                "Variation": v,
                "Weight_Impact": c_factor,
                "Status": status,
                **kpis
            })

    return pd.DataFrame(results_list)

In [11]:
# Setup your Base Scenario (e.g., Slow Chargers in Mild Weather)
base_config = {
    'start': start,
    'end': end,
    'alpha': [400.0] * 58,          # 58 chargers of 400kW
    'gamma': gama_fast,            # Your calculated gama_fast list
    'C_bat': [180.0] * 104,         # 104 buses with 180kWh
    'P': energy_price_LG,
    'delta_t': 1.0
}

# Run Sensitivity from -20% to +20%
# Here, we set weight_penalty to 0.12 (12% consumption increase for 100% battery increase)
results_df = run_comprehensive_sensitivity(
    base_config, 
    variations=[0.9, 1.0, 1.1],
    weight_penalty=0.008,  # Adjusted penalty to reflect a more realistic consumption increase
    peak_price=peak_price_LG
)

# Export results
results_df.to_excel("sensitivity_results_fast2.xlsx", index=False)

--- Running BASELINE (100%) ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
--- Running: Battery Capacity at 90% ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
--- Running: Battery Capacity at 110% ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
--- Running: Charger Power at 90% ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
--- Running: Charger Power at 110% ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
--- Running: Consumption Rate at 90% ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
--- Running: Consumption Rate at 110% ---
Problem size: 661 trips, 24 timesteps, 104 buses, 58 chargers
